# Phase 0: Model Verification

**Goal:** Verify that LLaMA 3.2 3B (4-bit quantized) loads on the target GPU,
runs inference, and produces curriculum-aligned output before any business logic is written.

**Target hardware:** Google Colab T4 (16 GB VRAM) or equivalent.

**Model:** `meta-llama/Llama-3.2-3B-Instruct` — 4-bit NF4 quantized via BitsAndBytes.

Run cells top-to-bottom. Expected total VRAM after load: ~3-4 GB.

## 1. Install Dependencies

In [ ]:
!pip install -q transformers accelerate bitsandbytes huggingface_hub

## 2. HuggingFace Login

Requires a Colab secret named `HF_API_KEY` with a token that has access to the gated LLaMA model.
Add it via **Runtime -> Manage Secrets** in Colab.

In [ ]:
from google.colab import userdata
from huggingface_hub import login

token = userdata.get('HF_API_KEY')
login(token)
print("Login successful")

## 3. Install PyTorch with CUDA 12.1

Colab T4 uses CUDA 12.x. Skip this cell if `torch.cuda.is_available()` already returns `True`.

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

## 4. GPU Availability Check

In [ ]:
import torch

print(f"PyTorch version : {torch.__version__}")
print(f"GPU available   : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU name        : {torch.cuda.get_device_name(0)}")
    print(f"VRAM total      : {props.total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError(
        "No GPU detected. Enable GPU runtime: Runtime -> Change runtime type -> T4 GPU"
    )

## 5. Load LLaMA 3.2 3B in 4-bit (NF4)

Config: `load_in_4bit=True`, `bnb_4bit_quant_type="nf4"`, `bnb_4bit_compute_dtype=torch.float16`.
Double quantization enabled to further reduce memory footprint.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

model.eval()
print("Model loaded successfully!")

## 6. VRAM Usage After Loading

Expected: ~3-4 GB allocated for the 3B model in 4-bit.
If allocated > 12 GB, something went wrong with the quant config.

In [ ]:
allocated = torch.cuda.memory_allocated() / 1e9
reserved  = torch.cuda.memory_reserved()  / 1e9
total     = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"VRAM allocated : {allocated:.2f} GB")
print(f"VRAM reserved  : {reserved:.2f} GB")
print(f"VRAM remaining : {total - reserved:.2f} GB")

## 7. Test Inference

Prompt: JSS2 Mathematics worksheet on fractions with Nigerian names and Naira.
Expected inference time on T4: ~10-20 s for 512 tokens.

In [ ]:
import time

messages = [
    {
        "role": "system",
        "content": (
            "You are an expert Nigerian educational content creator aligned with "
            "the NERDC curriculum. Always use Nigerian names (e.g. Chukwuemeka, "
            "Adaeze, Tunde, Ngozi) and Naira (\u20a6) in examples."
        ),
    },
    {
        "role": "user",
        "content": (
            "Generate a JSS2 Mathematics worksheet on fractions. "
            "Include 5 word problems using Nigerian names and Naira (\u20a6). "
            "Provide an answer key at the end."
        ),
    },
]

prompt = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

start = time.time()
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.7,
        do_sample=True,
        repetition_penalty=1.15,
    )
elapsed = time.time() - start

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
)

print(f"Inference time : {elapsed:.1f}s")
print(f"Tokens/s       : {512 / elapsed:.1f}")
print("-" * 60)
print(response)

## 8. Save Quantized Model to Google Drive

Saves the 4-bit quantized model weights to Drive so they can be loaded in future sessions
without re-downloading from HuggingFace (saves ~20-30 min per Colab session).

In [ ]:
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

SAVE_PATH = "/content/drive/MyDrive/enumverse_llm/llama3.2-3b-4bit"

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"Saved to {SAVE_PATH}")

## 9. (Optional) Load from Drive in Future Sessions

Use this cell instead of cell 5 in subsequent Colab sessions to skip the HuggingFace download.

In [ ]:
# from google.colab import drive
# from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
# import torch
#
# drive.mount('/content/drive')
#
# LOAD_PATH = "/content/drive/MyDrive/enumverse_llm/llama3.2-3b-4bit"
#
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.float16,
#     bnb_4bit_use_double_quant=True,
# )
#
# tokenizer = AutoTokenizer.from_pretrained(LOAD_PATH)
# model = AutoModelForCausalLM.from_pretrained(
#     LOAD_PATH,
#     quantization_config=bnb_config,
#     device_map="auto",
#     torch_dtype=torch.float16,
# )
# model.eval()
# print("Model loaded from Drive!")